# fastINFLECT 2.0 validation on the EXV BMV model

This notebook evaluates the literal schedule `25:100` on complete transformed marker distributions. It prints separate dip/IQR evidence, compares internal-validity and consensus diagnostics, and links the aggregate curve to the independent checkpointed modality audit. The curve is diagnostic only: it does not prove unimodality and does not by itself promote k = 50, 55, 62, or any other partition.

In [1]:
repo_path <- "/exports/para-lipg-hpc/mdmanurung/INFLECT"
model_path <- paste0(
  "/exports/para-lipg-hpc/Xuran/results/bmv/aurora_exvivo/cluster/",
  "model/xyf_som_model_with_LD_filter.rds"
)
audit_path <- file.path(
  repo_path,
  "inst/benchmarks/real-model-modality/modality-evidence.rds"
)
selected_audit_path <- file.path(
  repo_path,
  "inst/benchmarks/selected-k-modality/modality-evidence.rds"
)
selection_path <- file.path(
  repo_path,
  "inst/benchmarks/real-model-selection/selection-evidence.rds"
)

stopifnot(file.exists(file.path(repo_path, "DESCRIPTION")))
stopifnot(file.exists(model_path))
pkgload::load_all(repo_path, quiet = TRUE)

schedule <- 25L:100L
workers <- 2L
base_seed <- 42L
criterion <- "both"

cat("Literal requested schedule:\n")
print(schedule)
stopifnot(max(schedule) == 100L, length(schedule) == 76L)

read_status_kb <- function(field = "VmHWM") {
  status <- readLines("/proc/self/status", warn = FALSE)
  hit <- grep(paste0("^", field, ":"), status, value = TRUE)
  if (length(hit) == 0L) return(NA_real_)
  as.numeric(sub(".*?([0-9]+) kB.*", "\\1", hit[[1L]]))
}

package_version_or_na <- function(package) {
  if (!requireNamespace(package, quietly = TRUE)) return(NA_character_)
  as.character(utils::packageVersion(package))
}

code_identity <- list(
  git_commit = system2(
    "git",
    c("-C", repo_path, "rev-parse", "HEAD"),
    stdout = TRUE
  ),
  git_status = system2(
    "git",
    c("-C", repo_path, "status", "--short"),
    stdout = TRUE
  ),
  source_md5 = tools::md5sum(c(
    file.path(repo_path, "R/INFLECT.R"),
    file.path(repo_path, "R/iteration-QC.R"),
    file.path(repo_path, "R/inflect-qc-core.R")
  ))
)
print(code_identity)

Warning message:
“package ‘testthat’ was built under R version 4.5.2”


Literal requested schedule:
 [1]  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43
[20]  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62
[39]  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81
[58]  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 100
$git_commit
[1] "2390ae2974131beccd47950a95a165a8f1b0cf1a"

$git_status
  [1] " M .Rbuildignore"                                                               
  [2] " M .gitignore"                                                                  
  [3] " M DESCRIPTION"                                                                 
  [4] " M NAMESPACE"                                                                   
  [5] " M NEWS.md"                                                                     
  [6] " M R/FlowSOM-QC.R"                                                              
  [7] " M R/INFLECT.R"                          

In [2]:
load_timing <- system.time(model_exv <- readRDS(model_path))
view <- getFromNamespace("as_inflect_som", "fastINFLECT")(model_exv)

stopifnot(
  nrow(view$data) == 39050953L,
  view$map$nNodes == 900L,
  length(view$map$colsUsed) == 27L
)

model_contract <- list(
  events = nrow(view$data),
  nodes = view$map$nNodes,
  markers = view$prettyColnames[view$map$colsUsed],
  model_user_weights = view$inflect_source$user_weights,
  model_distance_weights = view$inflect_source$distance_weights,
  model_effective_layer_weights = view$inflect_source$effective_layer_weights,
  historical_intended_weights = c(X = 0.8, Y = 0.2),
  model_load_wall_seconds = unname(load_timing[["elapsed"]]),
  peak_rss_kb_after_load = read_status_kb()
)
print(model_contract)
cat(
  "The fitted model weights and historical intended 80/20 code space are ",
  "recorded as a sensitivity mismatch; the SOM is not retrained.\n",
  sep = ""
)

$events
[1] 39050953

$nodes
[1] 900

$markers
 [1] "CD45RA"       "TCRgd_CD141"  "CD8"          "CD3"          "CD19"        
 [6] "CD27"         "CD159c_NKG2C" "CD38"         "CD7"          "CD57"        
[11] "CD161"        "CD1c"         "HLA-DR"       "CD11c"        "CD25"        
[16] "CD16"         "CD185_CXCR5"  "CD14"         "CD303_TCRvd2" "CD4"         
[21] "CD294_CRTH2"  "CD21"         "CD56"         "FoxP3"        "Tbet"        
[26] "CD197_CCR7"   "CD127"       

$model_user_weights
layer1 layer2 
   0.5    0.5 

$model_distance_weights
     layer1      layer2 
0.001598314 3.855121262 

$model_effective_layer_weights
      layer1       layer2 
0.0007991569 1.9275606308 

$historical_intended_weights
  X   Y 
0.8 0.2 

$model_load_wall_seconds
[1] 67.869

$peak_rss_kb_after_load
[1] 11731712

The fitted model weights and historical intended 80/20 code space are recorded as a sensitivity mismatch; the SOM is not retrained.


In [3]:
run_warnings <- character()
run_timing <- system.time({
  result <- withCallingHandlers(
    INFLECT(
      FlowSOM.results = model_exv,
      set.i = schedule,
      multicore = TRUE,
      cores = workers,
      zeroes.in = TRUE,
      uniform.test = criterion,
      max.n.diptest = NULL,
      max.events.per.node = NULL,
      seed = base_seed,
      verbose = FALSE
    ),
    warning = function(w) {
      run_warnings <<- c(run_warnings, conditionMessage(w))
      invokeRestart("muffleWarning")
    }
  )
})

stopifnot(
  identical(names(result$metaclustering.list), as.character(schedule)),
  all(vapply(
    seq_along(schedule),
    function(i) length(unique(result$metaclustering.list[[i]])) == schedule[[i]],
    logical(1)
  ))
)

runtime_contract <- list(
  wall_seconds = unname(run_timing[["elapsed"]]),
  peak_rss_kb = read_status_kb(),
  warnings = unique(run_warnings),
  requested_workers = workers,
  effective_workers = result$provenance$effective_workers
)
print(runtime_contract)

$wall_seconds
[1] 716.814

$peak_rss_kb
[1] 27908172

$warnings
[1] "NaNs produced"

$requested_workers
[1] 2

$effective_workers
[1] 2



In [4]:
cat("Criterion and threshold contract:\n")
print(list(
  criterion = result$provenance$criterion,
  criterion_label = result$provenance$criterion_label,
  thresholds = result$provenance$thresholds,
  zero_handling_rule = result$provenance$zero_handling$rule,
  original_events = result$provenance$n_events,
  retained_events = result$provenance$n_events_retained,
  scoring_mode = result$provenance$scoring_mode,
  seeds = result$provenance$seeds,
  model = result$provenance$model
))

cat("Per-marker complete-distribution counts:\n")
print(result$provenance$zero_handling$per_marker)

cat("Separate dip, IQR, and combined summaries:\n")
print(result$provenance$criterion_summary)

cat("Fitted-versus-tested endpoint status:\n")
print(result$selection[c(
  "method",
  "k",
  "qc_pass_rate_at_k",
  "directly_tested",
  "partition_available",
  "k_status",
  "score_source"
)])

candidate_k <- intersect(c(
  27L, 28L, 38L, 44L, 50L, 55L, 60L, 62L, 65L
), schedule)
cat("Criterion matrices exist for named candidate partitions:\n")
print(lapply(as.character(candidate_k), function(k) {
  list(
    k = as.integer(k),
    dip = table(result$dip_pass[[k]], useNA = "ifany"),
    iqr = table(result$iqr_pass[[k]], useNA = "ifany"),
    combined = table(result$combined_pass[[k]], useNA = "ifany"),
    failures = table(result$qc.details[[k]]$failure_reason, useNA = "ifany")
  )
}))

Criterion and threshold contract:
$criterion
[1] "combined"

$criterion_label
[1] "dip and IQR QC pass rate"

$thresholds
$thresholds$dip_p_value
[1] 0.05

$thresholds$iqr
[1] 2


$zero_handling_rule
[1] "retain finite negative, zero, and positive transformed values"

$original_events
[1] 39050953

$retained_events
[1] 39050953

$scoring_mode
[1] "indexed_all_events"

$seeds
$seeds$base
[1] 42

$seeds$derivation
[1] "overflow-safe hash of subtree, marker, and RNG stream"


$model
$model$som_type
[1] "kohonen"

$model$som_class
[1] "kohonen"

$model$data_layer
[1] "layer1"

$model$code_layers
[1] "layer1" "layer2"

$model$user_weights
layer1 layer2 
   0.5    0.5 

$model$distance_weights
     layer1      layer2 
0.001598314 3.855121262 

$model$effective_layer_weights
      layer1       layer2 
0.0007991569 1.9275606308 


Per-marker complete-distribution counts:
         marker total_events finite_events negative_values zero_values
1          CD1c     39050953      39050953        164

## Cluster-number comparators

The next cell reads the checkpointed comparison run. Silhouette, Calinski-Harabasz, Davies-Bouldin, explained-dispersion knees, and consensus delta/stability are complementary diagnostics rather than interchangeable proofs of one correct biological k. The historical 80/20 and fitted-model code spaces are reported separately.

In [5]:
if (!file.exists(selection_path)) {
  stop(
    "Cluster-number comparison is incomplete. Run ",
    file.path(repo_path, "data-raw/evaluate-real-model-selection.R"),
    " before interpreting any k."
  )
}

selection_evidence <- readRDS(selection_path)
stopifnot(
  identical(selection_evidence$schedule, schedule),
  selection_evidence$n_events == 39050953L,
  selection_evidence$n_nodes == 900L,
  identical(selection_evidence$contract$zeroes_in, TRUE),
  is.na(selection_evidence$contract$max_n_diptest),
  is.na(selection_evidence$contract$max_events_per_node),
  identical(
    selection_evidence$full_inflect$derived_metric_provenance$version,
    "exclude_zero_event_nodes_v2"
  ),
  all(is.finite(
    selection_evidence$full_inflect$partition_metrics$explained_event_weighted
  )),
  length(selection_evidence$consensus_seed_provenance) == 3L
)

cat("Criterion-specific INFLECT selections:\n")
print(selection_evidence$full_inflect$criterion_selections)
cat("Internal-validity recommendations by code space:\n")
print(selection_evidence$full_inflect$metric_recommendations)
cat("Candidate-k internal metrics:\n")
print(subset(
  selection_evidence$full_inflect$partition_metrics,
  k %in% candidate_k
))
cat("Consensus delta-area/stability plateau:\n")
print(selection_evidence$consensus$plateau)
cat("Consensus diagnostics at candidate k:\n")
print(subset(selection_evidence$consensus$summary, k %in% candidate_k))
cat("Deterministic 5,000-event sampled IQR spread by materialised solution:\n")
print(selection_evidence$sample_iqr_summary)

Criterion-specific INFLECT selections:
      method  k qc_pass_rate_at_k directly_tested partition_available
1 inflection 28          42.98942            TRUE                TRUE
2    kneedle 44          54.46128            TRUE                TRUE
3  threshold NA                NA           FALSE               FALSE
4 inflection 38          74.56140            TRUE                TRUE
5    kneedle 55          75.82492            TRUE                TRUE
6  threshold NA                NA           FALSE               FALSE
7 inflection 27          37.44856            TRUE                TRUE
8    kneedle 44          46.71717            TRUE                TRUE
9  threshold NA                NA           FALSE               FALSE
          k_status score_source target unimodality_at_k error criterion
1 tested_partition       tested     NA         42.98942  <NA>       dip
2 tested_partition       tested     NA         54.46128  <NA>       dip
3    not_available         <NA>     95       

## Independent modality classifications

The fastINFLECT pass-rate curve is not converted into a modality claim. The next cell requires the checkpointed dip + ACR audit bundle. Benjamini-Hochberg correction is within the 27-marker family for each cluster, method, seed, and sample size; it is not global correction across clusters.

In [6]:
if (!file.exists(audit_path)) {
  stop(
    "Independent modality audit is incomplete. Run ",
    file.path(repo_path, "data-raw/submit-real-model-modality.sh"),
    " before interpreting any k."
  )
}

audit <- readRDS(audit_path)
if (!file.exists(selected_audit_path)) {
  stop(
    "Selected-k modality audit is incomplete. Run ",
    file.path(repo_path, "data-raw/submit-selected-k-modality.sh"),
    " before interpreting the dense-sweep selections."
  )
}
selected_audit <- readRDS(selected_audit_path)
stopifnot(
  isTRUE(audit$completeness$every_partition_materialized),
  isTRUE(audit$completeness$no_scan_above_100),
  audit$completeness$expected_base_combinations ==
    audit$completeness$observed_base_combinations,
  audit$completeness$expected_refinement_combinations ==
    audit$completeness$observed_refinement_combinations,
  isTRUE(selected_audit$completeness$every_partition_materialized),
  isTRUE(selected_audit$completeness$no_scan_above_100),
  selected_audit$completeness$expected_base_combinations ==
    selected_audit$completeness$observed_base_combinations,
  selected_audit$completeness$expected_refinement_combinations ==
    selected_audit$completeness$observed_refinement_combinations,
  identical(selected_audit$stage$source_provenance$selected_k, c(
    27L, 28L, 38L, 44L
  ))
)

all_classifications <- rbind(
  audit$classifications,
  selected_audit$classifications
)
all_solution_claims <- rbind(
  audit$solution_claims,
  selected_audit$solution_claims
)
cat("Independent modality classifications:\n")
print(with(all_classifications, table(solution_id, status)))
cat("Solution-level statements:\n")
print(all_solution_claims[c("solution_id", "k", "statement")])
cluster_occupancy <- unique(rbind(
  audit$stage$sample_manifest[c(
    "solution_id", "cluster", "cluster_events"
  )],
  selected_audit$stage$sample_manifest[c(
  "solution_id", "cluster", "cluster_events"
  )]
))
occupancy_summary <- do.call(rbind, lapply(
  split(cluster_occupancy, cluster_occupancy$solution_id),
  function(x) data.frame(
    solution_id = x$solution_id[[1L]],
    nominal_k = nrow(x),
    nonempty_event_clusters = sum(x$cluster_events > 0L),
    empty_event_clusters = sum(x$cluster_events == 0L),
    smallest_nonempty_cluster_events = min(x$cluster_events[x$cluster_events > 0L]),
    largest_cluster_event_fraction = max(x$cluster_events) / audit$stage$n_events
  )
))
cat("Nominal versus event-populated cluster counts:\n")
print(occupancy_summary)
cat("Refinement sensitivity:\n")
print(rbind(
  audit$refinement_sensitivity,
  selected_audit$refinement_sensitivity
))
cat("Audit wall time, peak memory, and completeness:\n")
print(list(
  primary = list(
    component_wall_seconds = audit$component_wall_seconds,
    component_wall_seconds_sum = audit$component_wall_seconds_sum,
    peak_rss_kb = audit$peak_rss_kb,
    completeness = audit$completeness
  ),
  selected_k = list(
    component_wall_seconds = selected_audit$component_wall_seconds,
    component_wall_seconds_sum = selected_audit$component_wall_seconds_sum,
    peak_rss_kb = selected_audit$peak_rss_kb,
    completeness = selected_audit$completeness
  )
))

Independent modality classifications:
                                status
solution_id                      ambiguous detected_multimodality
  flowsom_consensus_k62_seed1          231                     86
  flowsom_consensus_k62_seed2026       224                     93
  flowsom_consensus_k62_seed42         242                     85
  historical_80_20_ward_k50            139                     53
  historical_80_20_ward_k62            171                     62
  inflect_ward_k27                      96                     46
  inflect_ward_k28                      89                     42
  inflect_ward_k38                     127                     50
  inflect_ward_k44                     118                     56
  inflect_ward_k55                     169                     63
  inflect_ward_k60                     196                     66
  inflect_ward_k62                     211                     71
  inflect_ward_k65                     209                     76

In [7]:
synthesis_path <- file.path(
  repo_path,
  "inst/benchmarks/real-model-selection/bmv-k-selection-synthesis.rds"
)
stopifnot(file.exists(synthesis_path))
synthesis <- readRDS(synthesis_path)
cat("Final BMV k recommendation and sensitivity range:\n")
print(synthesis$decision)
cat("INFLECT criterion selections:\n")
print(synthesis$inflect_selections)
cat("Consensus comparison at the operational and spread k:\n")
print(synthesis$consensus)
cat("Curve-fit warning diagnostic:\n")
print(synthesis$curve_fit_warning[c("classification", "interpretation")])

dependency_versions <- c(
  R = as.character(getRversion()),
  fastINFLECT_source = unname(read.dcf(file.path(repo_path, "DESCRIPTION"))[1, "Version"]),
  FlowSOM = package_version_or_na("FlowSOM"),
  kohonen = package_version_or_na("kohonen"),
  diptest = package_version_or_na("diptest"),
  multimode = package_version_or_na("multimode"),
  ggplot2 = package_version_or_na("ggplot2")
)
print(dependency_versions)
print(result$provenance)
sessionInfo()

Final BMV k recommendation and sensitivity range:
$operational_k
[1] 44

$lower_resolution_inflection_k
[1] 27

$spread_sensitivity_upper_k
[1] 55

$status
[1] "Operational metaclustering recommendation, not a claim of biological truth or global unimodality."

INFLECT criterion selections:
$combined
inflection    kneedle 
        27         44 

$dip
inflection    kneedle 
        28         44 

$iqr_spread
inflection    kneedle 
        38         55 

Consensus comparison at the operational and spread k:
$plateau
                      method  k relative_delta_threshold minimum_seed_ari
1 first_5k_consensus_plateau 29                     0.01             0.95
  window_width
1            5

$operational_k
    k mean_cdf_area median_relative_delta_area min_seed_ari median_seed_ari
20 44     0.8681585                0.001224958    0.9986762       0.9986762

$spread_k
    k mean_cdf_area median_relative_delta_area min_seed_ari median_seed_ari
31 55     0.8742824               0.000429886

R version 4.5.1 (2025-06-13)
Platform: x86_64-conda-linux-gnu
Running under: Rocky Linux 8.10 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /exports/archive/hg-funcgenom-research/mdmanurung/conda/envs/R4_51/lib/libopenblasp-r0.3.29.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Europe/Amsterdam
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] fastINFLECT_2.0.0 testthat_3.3.2   

loaded via a namespace (and not attached):
  [1] kohonen_3.0.12              sandwich_3.1-1             
  [3] rlang_1.3.0                 magrittr_2.0.5             
  [5] multcomp_1.4-30             

## Interpretation guard

Only univariate marker marginals were assessed. Multivariate homogeneity and biological validity are out of scope. Report exact detected, ambiguous, and unresolved residuals. A strict solution-level “no detected multimodality” statement is allowed only when the audit contains no detected, ambiguous, or unresolved cluster-marker entry; never replace it with “truly unimodal.”